# A reconnection event

$J^2$ around the event at one time, and movies of $|J|^2$ and $|J_z|$ in the same window over time (the frames are
computed in parallel over the output times). The former notebook also had a copy of the single-sheet thickness
methods, which could not run there (its mask was never computed); that analysis is in `SingleCSAnalysis`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

from pta import config
from pta.fields import current_density
from pta.filtering import gaussian_fft_filter_2d
from pta.plotting import save_figure, show_field, use_style

use_style()
plt.rcParams.update({
    'font.size': 20,
    'axes.titlesize': 20,
    'axes.labelsize': 20,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'legend.fontsize': 20,
    'figure.titlesize': 25
})
RESULTS = config.results_dir("single_sheet", "MagRecEvent")

In [ ]:
BETA_P = 1
SNAPSHOT_TIME = 40
EVENT_CELL = (3550, 4043)             # centre of the window (cell)
WINDOW_CELLS = 400                    # half-width of the window
FILTER_FRACTION = 0.2                 # Gaussian filter width k0, as a fraction of the number of cells
MOVIE_LAST_TIME = 150                 # the movies run from the first output time to this one
MOVIE_FPS = 12
QUANTITIES = {                        # movie -> (label, colour range)
    "J_squared": (r"$|\mathbf{J}|^2$", (0, 0.5)),
    "J_par": (r"$|J_z|$", (0, 0.5)),
}

In [ ]:
run = config.run("beta_scan", beta_p=BETA_P)
dx, dy = run.dx, run.dy
corner = ((EVENT_CELL[0] - WINDOW_CELLS) * dx, (EVENT_CELL[1] - WINDOW_CELLS) * dy)


def patch(quantity, time):
    """|J|^2 or |J_z| (from the Gaussian-filtered B) in the window around the event at one output time."""
    filter_width = FILTER_FRACTION * run.shape[0]
    B = (gaussian_fft_filter_2d(component, k0=filter_width) for component in run.load_many(["Bx", "By", "Bz"], time))
    Jx, Jy, Jz = current_density(*B, dx, dy)
    window = (slice(EVENT_CELL[0] - WINDOW_CELLS, EVENT_CELL[0] + WINDOW_CELLS),
              slice(EVENT_CELL[1] - WINDOW_CELLS, EVENT_CELL[1] + WINDOW_CELLS))
    return (Jx[window]**2 + Jy[window]**2 + Jz[window]**2) if quantity == "J_squared" else np.abs(Jz[window])

In [ ]:
label, limits = QUANTITIES["J_squared"]
fig, ax = plt.subplots(figsize=(6, 6))
show_field(ax, patch("J_squared", SNAPSHOT_TIME), dx, dy, limits=limits, label=label, corner=corner)
ax.set_title(rf"$\beta_p={BETA_P}$, $t={SNAPSHOT_TIME}$")
save_figure(fig, RESULTS, f"J2_around_event_t{SNAPSHOT_TIME}")
plt.show()

In [ ]:
movie_times = [time for time in run.times if time <= MOVIE_LAST_TIME]

for quantity, (label, limits) in QUANTITIES.items():
    patches = run.map(lambda time: patch(quantity, time), times=movie_times)     # in parallel over the times

    fig, ax = plt.subplots(figsize=(6, 6))
    image = show_field(ax, patches[0], dx, dy, limits=limits, label=label, corner=corner)
    fig.tight_layout(pad=0.5)

    def update(frame_index):
        image.set_data(patches[frame_index].T)
        ax.set_title(rf"{label}   ($\beta_p={BETA_P}$,  $t={movie_times[frame_index]}$)")
        return [image]

    movie_path = RESULTS / f"{quantity}_patch_animation.mp4"
    FuncAnimation(fig, update, frames=len(patches), interval=1000 / MOVIE_FPS).save(str(movie_path), writer="ffmpeg", fps=MOVIE_FPS)
    plt.close(fig)
    print(f"saved {movie_path}")